# Notebook 01 - Dataset Preprocessing and Statistics

**Project:** Explainable Deep Learning for MRI Brain Tumor Classification and Segmentation Using Transfer Learning, MONAI, U-Net, and Grad-CAM

Run notebooks in order. Each notebook writes outputs into the same project folder so later notebooks can reuse them.


## Purpose

This notebook prepares the **2D brain tumor MRI classification dataset**. It:

1. Creates project folders.
2. Scans the classification dataset.
3. Standardizes class names.
4. Creates train/validation/test metadata.
5. Produces dataset statistics and sample-image figures.

Expected classification dataset format:

```text
brain_tumor_mri_dataset/
  Training/
    glioma/
    meningioma/
    pituitary/
    notumor/
  Testing/
    glioma/
    meningioma/
    pituitary/
    notumor/
```

You can use Kaggle or any similar folder-based MRI dataset.


In [ ]:
import sys, subprocess, os, json, random
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print("Running in Colab:", IN_COLAB)

# Install only in Colab. For local Jupyter, install these packages once in your environment.
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "torchvision", "scikit-learn", "pandas",
                           "numpy", "matplotlib", "pillow", "opencv-python", "tqdm"])


In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if IN_COLAB:
    PROJECT_ROOT = Path("/content/brain_tumor_xai_project")
    # Optional Drive path:
    # PROJECT_ROOT = Path("/content/drive/MyDrive/brain_tumor_xai_project")
else:
    PROJECT_ROOT = Path.cwd() / "brain_tumor_xai_project"

DATA_DIR = PROJECT_ROOT / "data"
CLASSIFICATION_DATA_ROOT = DATA_DIR / "brain_tumor_mri_dataset"
CLASSIFICATION_TRAIN_DIR = CLASSIFICATION_DATA_ROOT / "Training"
CLASSIFICATION_TEST_DIR = CLASSIFICATION_DATA_ROOT / "Testing"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURE_DIR = PROJECT_ROOT / "figures"

for p in [DATA_DIR, OUTPUT_DIR, MODEL_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Put classification dataset here:", CLASSIFICATION_DATA_ROOT)


## Optional: Google Drive and Kaggle setup

Uncomment these cells in Colab if you want to store data in Drive or download a Kaggle dataset. Kaggle requires your own `kaggle.json` API key.


In [ ]:
# Optional Google Drive
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
# Optional Kaggle download example
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/brain_tumor_xai_project/data --unzip


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

CLASS_NAME_MAP = {
    "glioma": "glioma",
    "glioma_tumor": "glioma",
    "meningioma": "meningioma",
    "meningioma_tumor": "meningioma",
    "pituitary": "pituitary",
    "pituitary_tumor": "pituitary",
    "notumor": "no_tumor",
    "no_tumor": "no_tumor",
    "normal": "no_tumor",
}

def standardize_class_name(name):
    key = name.strip().lower().replace("-", "_").replace(" ", "_")
    return CLASS_NAME_MAP.get(key, key)

def collect_images(split_dir, split_name):
    rows = []
    if not split_dir.exists():
        print(f"Warning: {split_dir} does not exist")
        return rows
    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        class_name = standardize_class_name(class_dir.name)
        for img_path in class_dir.rglob("*"):
            if img_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({
                    "path": str(img_path),
                    "class_name": class_name,
                    "original_class_folder": class_dir.name,
                    "split": split_name
                })
    return rows

rows = []
rows += collect_images(CLASSIFICATION_TRAIN_DIR, "train")
rows += collect_images(CLASSIFICATION_TEST_DIR, "test")

if not rows:
    raise FileNotFoundError("No images found. Set CLASSIFICATION_DATA_ROOT or put dataset in the expected folder.")

df = pd.DataFrame(rows)
class_names = sorted(df["class_name"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(class_names)}
df["label"] = df["class_name"].map(class_to_idx)

print("Total images:", len(df))
print("Classes:", class_names)
df.head()


In [ ]:
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

train_part, val_part = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label"],
    random_state=SEED
)
train_part = train_part.copy(); train_part["split"] = "train"
val_part = val_part.copy(); val_part["split"] = "val"

metadata = pd.concat([train_part, val_part, test_df], ignore_index=True)
metadata = metadata.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(metadata.groupby(["split", "class_name"]).size().unstack(fill_value=0))


In [ ]:
metadata_path = OUTPUT_DIR / "classification_metadata.csv"
class_path = OUTPUT_DIR / "class_names.json"
config_path = PROJECT_ROOT / "project_config.json"

metadata.to_csv(metadata_path, index=False)
class_path.write_text(json.dumps(class_names, indent=2))

config = {
    "project_root": str(PROJECT_ROOT),
    "classification_metadata_csv": str(metadata_path),
    "class_names_json": str(class_path),
    "model_dir": str(MODEL_DIR),
    "figure_dir": str(FIGURE_DIR),
    "image_size": 224,
    "seed": SEED
}
config_path.write_text(json.dumps(config, indent=2))

print("Saved metadata:", metadata_path)
print("Saved config:", config_path)


In [ ]:
counts = metadata.groupby(["split", "class_name"]).size().unstack(fill_value=0)
ax = counts.plot(kind="bar", figsize=(10, 5))
ax.set_title("Classification Dataset Distribution")
ax.set_ylabel("Number of images")
ax.set_xlabel("Split")
plt.xticks(rotation=0)
plt.tight_layout()
fig_path = FIGURE_DIR / "classification_class_distribution.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", fig_path)


In [ ]:
samples = metadata.groupby("class_name").sample(n=1, random_state=SEED)
plt.figure(figsize=(12, 4))
for i, (_, row) in enumerate(samples.iterrows(), start=1):
    img = Image.open(row["path"]).convert("RGB")
    plt.subplot(1, len(samples), i)
    plt.imshow(img, cmap="gray")
    plt.title(row["class_name"])
    plt.axis("off")
plt.tight_layout()
fig_path = FIGURE_DIR / "classification_sample_images.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", fig_path)


## Outputs from Notebook 01

- `classification_metadata.csv`
- `class_names.json`
- `project_config.json`
- `classification_class_distribution.png`
- `classification_sample_images.png`

Next: run `02_classification_training.ipynb`.
